# 00 — Baseline Supervised Dataset Builder (No Feature Engineering)

Creates `processed_data/supervised_baseline_3h.csv` using **only time-t features** (HOOD + time + weather).

- No lags/rolling
- Label is `y_class` from next 3-hour block (t+1): 0 / 1 / 2+


In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"

MODEL_PATH = DATA_DIR / "model_hood_3h_weather.csv"
WEATHER3H_PATH = DATA_DIR / "weather_3h.csv"
OUT_BASELINE = DATA_DIR / "supervised_baseline_3h.csv"

FREQ = "3h"
HIGH_RISK_THRESHOLD = 2

In [4]:
base = pd.read_csv(MODEL_PATH, low_memory=False)
base["time_3h"] = pd.to_datetime(base["time_3h"], errors="coerce")
base["HOOD_158_CODE"] = pd.to_numeric(base["HOOD_158_CODE"], errors="coerce")
base = base.dropna(subset=["time_3h","HOOD_158_CODE"]).copy()
base["HOOD_158_CODE"] = base["HOOD_158_CODE"].astype(int).astype(str).str.zfill(3)

keep_collision = ["collisions"]
for c in ["injury_collisions","ftr_collisions","pd_collisions","pedestrian_collisions","bicycle_collisions"]:
    if c in base.columns:
        keep_collision.append(c)

base = base[["HOOD_158_CODE","time_3h"] + keep_collision].copy()

w = pd.read_csv(WEATHER3H_PATH, low_memory=False)
w["time_3h"] = pd.to_datetime(w["time_3h"], errors="coerce")
w = w.dropna(subset=["time_3h"]).sort_values("time_3h").drop_duplicates("time_3h", keep="first")

print("Base:", base.shape, "HOODs:", base["HOOD_158_CODE"].nunique())
print("Weather:", w.shape, "Time:", w["time_3h"].min(), "→", w["time_3h"].max())


Base: (148208, 8) HOODs: 158
Weather: (8768, 10) Time: 2023-01-01 00:00:00 → 2025-12-31 21:00:00


In [5]:
hoods = sorted(base["HOOD_158_CODE"].unique())
times = pd.date_range(w["time_3h"].min(), w["time_3h"].max(), freq=FREQ)
grid = pd.MultiIndex.from_product([hoods, times], names=["HOOD_158_CODE","time_3h"]).to_frame(index=False)

df = grid.merge(base, on=["HOOD_158_CODE","time_3h"], how="left")
for c in keep_collision:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype("int32")

df = df.merge(w, on="time_3h", how="left", validate="m:1")
for c in w.columns:
    if c != "time_3h" and c in df.columns and df[c].dtype.kind in "fc":
        df[c] = df[c].interpolate(limit_direction="both")

df = df.sort_values(["HOOD_158_CODE","time_3h"]).reset_index(drop=True)
print("Grid:", df.shape, "Zero share:", round(float((df['collisions']==0).mean()), 4))


Grid: (1385344, 17) Zero share: 0.893


In [6]:
df["block_hour"] = df["time_3h"].dt.hour
df["dow_num"] = df["time_3h"].dt.dayofweek
df["month_num"] = df["time_3h"].dt.month
df["is_weekend"] = (df["dow_num"] >= 5).astype("uint8")

# cyclical time encodings
import numpy as np

df["hour_sin"] = np.sin(2*np.pi*df["block_hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["block_hour"]/24)
df["dow_sin"]  = np.sin(2*np.pi*df["dow_num"]/7)
df["dow_cos"]  = np.cos(2*np.pi*df["dow_num"]/7)
df["month_sin"] = np.sin(2*np.pi*df["month_num"]/12)
df["month_cos"] = np.cos(2*np.pi*df["month_num"]/12)


In [7]:
g = df.groupby("HOOD_158_CODE", sort=False)
df["y_count_next"] = g["collisions"].shift(-1)

df["y_class"] = np.select(
    [df["y_count_next"].isna(), df["y_count_next"]==0, df["y_count_next"]==1, df["y_count_next"]>=HIGH_RISK_THRESHOLD],
    [np.nan, 0, 1, 2],
    default=np.nan
)

df = df[df["y_class"].notna()].copy()
df["y_class"] = df["y_class"].astype("int8")

print("Rows:", df.shape)
print("Class dist:", df["y_class"].value_counts(normalize=True).sort_index().round(4).to_dict())


Rows: (1385186, 29)
Class dist: {0: 0.893, 1: 0.0927, 2: 0.0143}


In [8]:
# drop any accidental history columns
bad = [c for c in df.columns if c.lower().startswith("coll_lag_") or c.lower().startswith("coll_roll_") or c.lower().endswith("_lag_1")]
if bad:
    df = df.drop(columns=bad)
    print("Dropped:", bad)

df.to_csv(OUT_BASELINE, index=False, float_format="%.3f")
print("Saved:", OUT_BASELINE)
df.head()


Saved: ..\data\processed\supervised_baseline_3h.csv


,HOOD_158_CODE,time_3h,collisions,injury_collisions,ftr_collisions,pd_collisions,pedestrian_collisions,bicycle_collisions,pressure_sea,wind_speed,...,month_num,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos,month_sin,month_cos,y_count_next,y_class
0,001,2023-01-01 00:00:00,1,1,1,0,0,0,101.190,11.333,...,1,1,0.000000e+00,1.000000e+00,-0.781831,0.62349,0.5,0.866025,0.0,0
1,001,2023-01-01 03:00:00,0,0,0,0,0,0,101.317,11.667,...,1,1,7.071068e-01,7.071068e-01,-0.781831,0.62349,0.5,0.866025,0.0,0
2,001,2023-01-01 06:00:00,0,0,0,0,0,0,101.477,20.000,...,1,1,1.000000e+00,6.123234e-17,-0.781831,0.62349,0.5,0.866025,0.0,0
3,001,2023-01-01 09:00:00,0,0,0,0,0,0,101.503,9.000,...,1,1,7.071068e-01,-7.071068e-01,-0.781831,0.62349,0.5,0.866025,1.0,1
4,001,2023-01-01 12:00:00,1,0,1,0,0,0,101.410,10.000,...,1,1,1.224647e-16,-1.000000e+00,-0.781831,0.62349,0.5,0.866025,0.0,0
